# 49 — Region-Constrained Design Search v2

**Subtitle:** Topology Specifies Robust Engineering

**Notebook purpose**

Search over admissible regions first. Select operating points second.

Notebook 37 introduced operating regimes.  
Notebook 43 developed robustness as preservation of the largest connected admissible region.  
Notebook 49 turns that idea into a reusable design-search procedure.

\[
\boxed{
\text{Design}
\rightarrow
\Omega
\rightarrow
\Omega_c
\rightarrow
d(\partial\Omega_c)
\rightarrow
x^\*
\rightarrow
\text{Fault-Tolerant Computation}
}
\]

## Setup

This notebook creates figures, CSV/JSON tables, and a downloadable ZIP bundle.

```text
figures/
results/csv/
results/json/
results/49_outputs_v2.zip
```

In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import deque
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

def save_table(df, stem):
    csv_path = CSV / f"{stem}.csv"
    json_path = JS / f"{stem}.json"
    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient="records", indent=2)
    print(f"saved: {csv_path}")
    print(f"saved: {json_path}")

def grid(n=220):
    x = np.linspace(0, 1, n)
    y = np.linspace(0, 1, n)
    X, Y = np.meshgrid(x, y)
    return x, y, X, Y

## 1. Reusable design object

Notebook 49 v2 uses a `RegionDesign` dataclass so candidate architectures are evaluated consistently.

In [ ]:
@dataclass
class RegionDesign:
    name: str
    drive_shift: float = 0.0
    loss_scale: float = 1.0
    timing_scale: float = 1.0
    calibration_error: float = 0.0
    routing_noise: float = 0.0
    detection_support: float = 1.0
    coupling_support: float = 1.0
    hardware_cost: float = 1.0
    control_complexity: float = 1.0

def admissibility_score(X, Y, design: RegionDesign):
    drive = np.clip(X + design.drive_shift, 0, 1.2)
    loss = np.clip(Y * design.loss_scale, 0, 1.4)
    enough_drive = 1 / (1 + np.exp(-18 * (drive - 0.30)))
    loss_survival = np.exp(-2.8 * loss)
    overdrive = np.exp(-9.0 * np.maximum(drive - 0.86, 0) ** 2)
    timing = np.exp(-design.timing_scale * 5.0 * np.maximum(drive + 0.74 * loss - 1.08, 0) ** 2)
    calibration = np.exp(-4.0 * design.calibration_error ** 2)
    broad_region = np.exp(-((drive - 0.62) ** 2 / 0.27 + (loss - 0.20) ** 2 / 0.18))
    routing_defects = 1 - design.routing_noise * np.sin(10 * X) * np.sin(8 * Y)
    score = design.coupling_support * design.detection_support * enough_drive * loss_survival * overdrive * timing * calibration * (0.50 + 0.50 * broad_region) * routing_defects
    score = np.clip(score, 0, None)
    return score / (score.max() + 1e-12)

designs = [
    RegionDesign("single cavity", drive_shift=-0.06, loss_scale=1.28, timing_scale=1.20, calibration_error=0.08, routing_noise=0.06, detection_support=0.92, coupling_support=0.94, hardware_cost=2, control_complexity=2),
    RegionDesign("coupled resonators", drive_shift=-0.02, loss_scale=1.10, timing_scale=1.05, calibration_error=0.05, routing_noise=0.04, detection_support=0.95, coupling_support=1.00, hardware_cost=4, control_complexity=4),
    RegionDesign("programmable mesh", drive_shift=0.02, loss_scale=1.00, timing_scale=0.90, calibration_error=0.04, routing_noise=0.02, detection_support=0.98, coupling_support=1.03, hardware_cost=7, control_complexity=7),
    RegionDesign("hybrid chip", drive_shift=0.04, loss_scale=0.82, timing_scale=0.72, calibration_error=0.02, routing_noise=0.01, detection_support=1.02, coupling_support=1.08, hardware_cost=9, control_complexity=8),
]
pd.DataFrame([asdict(d) for d in designs])

## 2. Reusable topology helpers

The search computes \(\Omega\), \(\Omega_c\), \(d(\partial\Omega_c)\), and \(x^\*\).

In [ ]:
def connected_components(mask):
    visited = np.zeros_like(mask, dtype=bool)
    h, w = mask.shape
    comps = []
    for i in range(h):
        for j in range(w):
            if mask[i, j] and not visited[i, j]:
                q = deque([(i, j)])
                visited[i, j] = True
                pts = []
                while q:
                    a, b = q.popleft()
                    pts.append((a, b))
                    for da, db in [(1,0), (-1,0), (0,1), (0,-1)]:
                        na, nb = a + da, b + db
                        if 0 <= na < h and 0 <= nb < w and mask[na, nb] and not visited[na, nb]:
                            visited[na, nb] = True
                            q.append((na, nb))
                comps.append(pts)
    comps.sort(key=len, reverse=True)
    return comps

def largest_component_mask(mask):
    comps = connected_components(mask)
    largest = np.zeros_like(mask, dtype=bool)
    if comps:
        pts = np.array(comps[0])
        largest[pts[:,0], pts[:,1]] = True
    return largest, comps

def boundary_distance(mask):
    h, w = mask.shape
    INF = h + w + 10
    dist = np.full((h, w), INF, dtype=float)
    q = deque()
    for i in range(h):
        for j in range(w):
            if not mask[i, j]:
                dist[i, j] = 0
                q.append((i, j))
    while q:
        i, j = q.popleft()
        for di, dj in [(1,0), (-1,0), (0,1), (0,-1)]:
            ni, nj = i + di, j + dj
            if 0 <= ni < h and 0 <= nj < w and dist[ni, nj] > dist[i, j] + 1:
                dist[ni, nj] = dist[i, j] + 1
                q.append((ni, nj))
    return dist

def evaluate_region(design: RegionDesign, n=220, threshold=0.50):
    x, y, X, Y = grid(n)
    Z = admissibility_score(X, Y, design)
    mask = Z >= threshold
    largest, comps = largest_component_mask(mask)
    dist = boundary_distance(largest)
    if comps:
        pts = np.array(comps[0])
        largest_fraction = len(comps[0]) / mask.size
        admitted_area = mask.mean()
        avg_margin = dist[pts[:,0], pts[:,1]].mean() / max(mask.shape)
        max_margin = dist[pts[:,0], pts[:,1]].max() / max(mask.shape)
        best = pts[np.argmax(dist[pts[:,0], pts[:,1]])]
        x_star = x[best[1]]
        y_star = y[best[0]]
    else:
        largest_fraction = admitted_area = avg_margin = max_margin = x_star = y_star = 0.0
    return {
        "design": design.name,
        "admitted_area": admitted_area,
        "largest_component": largest_fraction,
        "component_count": len(comps),
        "average_margin": avg_margin,
        "maximum_margin": max_margin,
        "x_star": x_star,
        "y_star": y_star,
        "hardware_cost": design.hardware_cost,
        "control_complexity": design.control_complexity,
        "x": x, "y": y, "X": X, "Y": Y, "Z": Z, "mask": mask, "largest": largest, "dist": dist,
    }

## 3. Engineering specification chain

Make \(\Omega_c\) visually dominant. Everything in the search serves the largest connected admissible component.

In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 4.9))
ax.axis("off")
items = [
    ("Design", "candidate"),
    ("Ω", "admissible region"),
    ("Ωc", "largest connected\nadmissible region"),
    ("d(∂Ωc)", "distance transform"),
    ("x*", "maximum-margin point"),
    ("Fault-Tolerant\nComputation", "execution"),
]
xs = np.linspace(0.08, 0.92, len(items))
y = 0.56
w, h = 0.135, 0.28
for i, (title, subtitle) in enumerate(items):
    is_core = title == "Ωc"
    ww = w * (1.28 if is_core else 1.0)
    hh = h * (1.16 if is_core else 1.0)
    lw = 3.2 if is_core else 2.1
    ax.add_patch(Rectangle((xs[i]-ww/2, y-hh/2), ww, hh, fill=False, linewidth=lw))
    ax.text(xs[i], y+0.055, title, ha="center", va="center", fontsize=14 if is_core else 12, weight="bold")
    ax.text(xs[i], y-0.075, subtitle, ha="center", va="center", fontsize=8.8)
    if i < len(items)-1:
        next_core = items[i+1][0] == "Ωc"
        next_w = w * (1.28 if next_core else 1.0)
        ax.annotate("", xy=(xs[i+1]-next_w/2-0.010, y), xytext=(xs[i]+ww/2+0.010, y), arrowprops=dict(arrowstyle="->", linewidth=2.2))
ax.text(0.5, 0.16, "Preserve the region before selecting the point.", ha="center", fontsize=12)
ax.set_title("Region-Constrained Design Search Specification", fontsize=18)
savefig(fig, "49_v2_01_design_specification.png")
plt.show()

## 4. Point search versus region search

Region search optimizes admissible topology before selecting a point.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax in axes:
    ax.axis("off")
left = [("candidate", .82), ("score", .62), ("gradient", .42), ("repeat", .22)]
right = [("candidate", .86), ("simulate", .71), ("Ω", .56), ("Ωc", .41), ("distance", .26), ("x*", .11)]
for label, yy in left:
    axes[0].add_patch(Rectangle((.31, yy-.055), .38, .11, fill=False, linewidth=2))
    axes[0].text(.5, yy, label, ha="center", va="center", fontsize=11)
for i in range(len(left)-1):
    axes[0].annotate("", xy=(.5, left[i+1][1]+.06), xytext=(.5, left[i][1]-.06), arrowprops=dict(arrowstyle="->", linewidth=2))
axes[0].set_title("Point search", fontsize=15)
axes[0].text(.5, .03, "optimizes a local score", ha="center", fontsize=11)
for label, yy in right:
    axes[1].add_patch(Rectangle((.30, yy-.045), .40, .09, fill=False, linewidth=2))
    axes[1].text(.5, yy, label, ha="center", va="center", fontsize=11)
for i in range(len(right)-1):
    axes[1].annotate("", xy=(.5, right[i+1][1]+.05), xytext=(.5, right[i][1]-.05), arrowprops=dict(arrowstyle="->", linewidth=2))
axes[1].set_title("Region search", fontsize=15)
axes[1].text(.5, .03, "optimizes admissible topology first", ha="center", fontsize=11)
fig.suptitle("Point Search vs Region Search", fontsize=18)
savefig(fig, "49_v2_02_point_vs_region_search.png")
plt.show()

## 5. Region objective

\[
\boxed{
J(D)=
\alpha |\Omega_c(D)|
+\beta M(D)
-\gamma C(D)
-\delta K(D)
}
\]

First maximize the region. Then choose \(x^\*\).

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 5.3))
ax.axis("off")
ax.text(0.5, 0.72, r"$J(D)=\alpha|\Omega_c(D)|+\beta M(D)-\gamma C(D)-\delta K(D)$", ha="center", fontsize=23)
ax.add_patch(Rectangle((0.13, 0.62), 0.74, 0.17, fill=False, linewidth=2.2))
terms = [
    ("+ |Ωc|", "region size", .18),
    ("+ M", "robustness margin", .38),
    ("− C", "hardware cost", .60),
    ("− K", "control complexity", .80),
]
for sym, meaning, x0 in terms:
    ax.add_patch(Rectangle((x0-.095, .30), .19, .20, fill=False, linewidth=2))
    ax.text(x0, .43, sym, ha="center", va="center", fontsize=18, weight="bold")
    ax.text(x0, .34, meaning, ha="center", va="center", fontsize=10)
ax.annotate("maximize region", xy=(0.40, 0.18), xytext=(0.30, 0.18), arrowprops=dict(arrowstyle="->", linewidth=2), fontsize=12)
ax.annotate("then choose x*", xy=(0.62, 0.18), xytext=(0.48, 0.18), arrowprops=dict(arrowstyle="->", linewidth=2), fontsize=12)
ax.set_title("Region-Constrained Objective Function", fontsize=18)
savefig(fig, "49_v2_03_region_objective.png")
plt.show()

## 6. Candidate architecture evaluation

The table is generated from region metrics, not handwritten scores.

In [ ]:
alpha, beta, gamma, delta = 1.0, 1.4, 0.018, 0.014
evaluations = [evaluate_region(d) for d in designs]
rows = []
for design, ev in zip(designs, evaluations):
    score = alpha * ev["largest_component"] + beta * ev["maximum_margin"] - gamma * design.hardware_cost - delta * design.control_complexity
    rows.append({
        "design": design.name,
        "admitted_area": ev["admitted_area"],
        "largest_component": ev["largest_component"],
        "component_count": ev["component_count"],
        "maximum_margin": ev["maximum_margin"],
        "average_margin": ev["average_margin"],
        "hardware_cost": design.hardware_cost,
        "control_complexity": design.control_complexity,
        "score": score,
        "x_star": ev["x_star"],
        "y_star": ev["y_star"],
    })
architecture_metrics = pd.DataFrame(rows).sort_values("score", ascending=False)
save_table(architecture_metrics, "49_v2_architecture_metrics")
architecture_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.7))
ax.axis("off")
display_df = architecture_metrics.copy()
for col in ["admitted_area", "largest_component", "maximum_margin", "average_margin", "score", "x_star", "y_star"]:
    display_df[col] = display_df[col].map(lambda v: f"{v:.3f}")
table = ax.table(cellText=display_df.values, colLabels=display_df.columns, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(8.4)
table.scale(1.02, 1.55)
for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.0)
    if r == 0:
        cell.set_text_props(weight="bold")
    if c == 0 and r > 0:
        cell.set_text_props(ha="left")
ax.set_title("Architecture Metrics from Region-Constrained Search", fontsize=17, pad=18)
savefig(fig, "49_v2_04_architecture_metrics_table.png")
plt.show()

## 7. Design evolution

The search is successful when the largest connected region and maximum margin both increase.

In [ ]:
iteration_designs = [
    RegionDesign("initial", drive_shift=-0.09, loss_scale=1.35, timing_scale=1.25, calibration_error=0.08, routing_noise=0.08, detection_support=0.92, coupling_support=0.92, hardware_cost=2, control_complexity=2),
    RegionDesign("admitted", drive_shift=-0.04, loss_scale=1.20, timing_scale=1.12, calibration_error=0.06, routing_noise=0.05, detection_support=0.95, coupling_support=0.98, hardware_cost=3, control_complexity=3),
    RegionDesign("connected", drive_shift=0.00, loss_scale=1.02, timing_scale=0.95, calibration_error=0.04, routing_noise=0.025, detection_support=0.99, coupling_support=1.02, hardware_cost=5, control_complexity=5),
    RegionDesign("expanded", drive_shift=0.04, loss_scale=0.84, timing_scale=0.76, calibration_error=0.02, routing_noise=0.01, detection_support=1.02, coupling_support=1.08, hardware_cost=8, control_complexity=7),
]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.25), sharex=True, sharey=True)
evo_rows = []
for ax, d in zip(axes, iteration_designs):
    ev = evaluate_region(d, n=185)
    ax.imshow(ev["Z"], origin="lower", extent=[0,1,0,1], vmin=0, vmax=1, aspect="auto")
    ax.contour(ev["X"], ev["Y"], ev["Z"], levels=[0.50], colors="black", linewidths=2.1)
    ax.scatter([ev["x_star"]], [ev["y_star"]], s=120, color="red", zorder=4)
    ax.set_title(d.name)
    ax.set_xlabel("drive")
    if ax is axes[0]:
        ax.set_ylabel("loss")
    ax.text(0.05, 0.88, f"$A$={ev['largest_component']:.2f}", transform=ax.transAxes, fontsize=11, weight="bold")
    ax.text(0.05, 0.77, f"$M$={ev['maximum_margin']:.2f}", transform=ax.transAxes, fontsize=11, weight="bold")
    evo_rows.append({"iteration": d.name, "largest_component": ev["largest_component"], "maximum_margin": ev["maximum_margin"], "component_count": ev["component_count"]})
evolution = pd.DataFrame(evo_rows)
save_table(evolution, "49_v2_design_evolution")
fig.suptitle("Design Evolution: Expand Ωc Before Selecting x*", fontsize=17, y=1.03)
savefig(fig, "49_v2_05_design_evolution.png")
plt.show()
evolution

## 8. Distance transform

The distance transform selects the maximum-margin operating point \(x^\*\).

In [ ]:
best_name = architecture_metrics.iloc[0]["design"]
best_design = next(d for d in designs if d.name == best_name)
best_eval = evaluate_region(best_design, n=250)
dist_norm = best_eval["dist"] / (best_eval["dist"].max() + 1e-12)
dist_masked = np.where(best_eval["largest"], dist_norm, np.nan)
fig, ax = plt.subplots(figsize=(10, 6.2))
im = ax.imshow(dist_masked, origin="lower", extent=[0,1,0,1], aspect="auto", vmin=0, vmax=1)
ax.contour(best_eval["X"], best_eval["Y"], best_eval["Z"], levels=[0.50], colors="black", linewidths=2.2)
ax.scatter([best_eval["x_star"]], [best_eval["y_star"]], s=250, color="red", zorder=4)
ax.annotate("maximum robustness margin", xy=(best_eval["x_star"], best_eval["y_star"]),
            xytext=(best_eval["x_star"]+0.10, best_eval["y_star"]+0.19),
            arrowprops=dict(arrowstyle="->", linewidth=2.0), fontsize=11)
ax.set_title("Distance Transform Selects x*", fontsize=17)
ax.set_xlabel("drive")
ax.set_ylabel("loss")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("normalized distance to failure boundary")
savefig(fig, "49_v2_06_distance_transform.png")
plt.show()

## 9. Search algorithm

Short verbs make the search read like a compiler pipeline.

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 7.0))
ax.axis("off")
steps = ["Generate", "Simulate", "Threshold", "Connect", "Distance", "Score", "Select"]
ys = np.linspace(0.88, 0.12, len(steps))
for i, (step, yy) in enumerate(zip(steps, ys)):
    ax.add_patch(Rectangle((0.31, yy-0.045), 0.38, 0.09, fill=False, linewidth=2.1))
    ax.text(0.50, yy, step, ha="center", va="center", fontsize=12, weight="bold")
    if i < len(steps)-1:
        ax.annotate("", xy=(0.5, ys[i+1]+0.052), xytext=(0.5, yy-0.052),
                    arrowprops=dict(arrowstyle="->", linewidth=2.1))
ax.set_title("Region-Constrained Search Algorithm", fontsize=17)
savefig(fig, "49_v2_07_search_algorithm.png")
plt.show()

## 10. Math-style pseudocode

The algorithm searches over designs \(D\), not points.

In [ ]:
algorithm_lines = [
    r"for D ∈ Designs:",
    r"    Ω(D) ← admissible(D)",
    r"    Ωc(D) ← largest_component(Ω)",
    r"    M(D) ← max_distance(Ωc, ∂Ωc)",
    r"    J(D) ← α|Ωc| + βM − γC − δK",
    r"return argmax_D J(D)",
    r"x* ← maximum_margin_point(Ωc)",
]
fig, ax = plt.subplots(figsize=(11.5, 5.0))
ax.axis("off")
ax.add_patch(Rectangle((0.08, 0.10), 0.84, 0.78, fill=False, linewidth=2.0))
for i, line in enumerate(algorithm_lines):
    ax.text(0.13, 0.80 - i*0.10, line, family="monospace", fontsize=13)
ax.set_title("Algorithm: Search Over Regions, Not Points", fontsize=17)
savefig(fig, "49_v2_08_math_algorithm.png")
plt.show()

## 11. Search progress

Shading between the curves emphasizes robustness expansion.

In [ ]:
iterations = np.arange(1, len(evolution)+1)
fig, ax = plt.subplots(figsize=(8.8, 5.4))
ax.plot(iterations, evolution["largest_component"], marker="o", linewidth=2.6, label="largest connected region")
ax.plot(iterations, evolution["maximum_margin"], marker="s", linewidth=2.6, label="maximum margin")
ax.fill_between(iterations, evolution["maximum_margin"], evolution["largest_component"], alpha=0.18)
ax.set_xticks(iterations)
ax.set_xticklabels(evolution["iteration"])
ax.set_ylim(0, max(evolution["largest_component"].max(), evolution["maximum_margin"].max())*1.25)
ax.set_title("Topology Improves Before Operating Margin", fontsize=17)
ax.set_ylabel("relative metric")
ax.grid(alpha=0.25)
ax.legend()
savefig(fig, "49_v2_09_search_progress.png")
plt.show()

## 12. Engineering tradeoffs

Radar charts were removed. Horizontal bars make comparisons faster.

In [ ]:
trade = architecture_metrics.copy()
trade["region"] = trade["largest_component"] / trade["largest_component"].max()
trade["margin"] = trade["maximum_margin"] / trade["maximum_margin"].max()
trade["cost"] = trade["hardware_cost"] / trade["hardware_cost"].max()
trade["complexity"] = trade["control_complexity"] / trade["control_complexity"].max()
metrics = ["region", "margin", "cost", "complexity"]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.6), sharey=True)
for ax, metric in zip(axes, metrics):
    ordered = trade.sort_values(metric)
    ax.barh(ordered["design"], ordered[metric])
    ax.set_xlim(0, 1.05)
    ax.set_title(metric)
    ax.grid(axis="x", alpha=0.25)
fig.suptitle("Engineering Tradeoffs: Region and Margin vs Cost and Complexity", fontsize=17, y=1.03)
savefig(fig, "49_v2_10_tradeoff_bars.png")
plt.show()

## 13. Region search pipeline

Use “Region Objective” to match the objective function.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.9))
ax.axis("off")
pipeline = [
    ("Physics", "what can exist?"),
    ("Resources", "what was generated?"),
    ("Admissibility", "what survives?"),
    ("Topology", "Ωc"),
    ("Distance\nGeometry", "d(∂Ωc)"),
    ("Region\nObjective", "J(D)"),
    ("Operating\nPoint", "x*"),
    ("Fault-Tolerant\nComputation", "execution"),
]
xs = np.linspace(0.06, 0.94, len(pipeline))
y = 0.55
w, h = 0.105, 0.28
for i, (title, subtitle) in enumerate(pipeline):
    ax.add_patch(Rectangle((xs[i]-w/2, y-h/2), w, h, fill=False, linewidth=2.0))
    ax.text(xs[i], y+0.045, title, ha="center", va="center", fontsize=9.3, weight="bold")
    ax.text(xs[i], y-0.070, subtitle, ha="center", va="center", fontsize=8.0)
    if i < len(pipeline)-1:
        ax.annotate("", xy=(xs[i+1]-w/2-0.007, y), xytext=(xs[i]+w/2+0.007, y),
                    arrowprops=dict(arrowstyle="->", linewidth=2.0))
ax.text(0.5, 0.17, "Topology specifies robustness. Robustness specifies operation. Operation specifies computation.",
        ha="center", fontsize=12)
ax.set_title("Region Search Pipeline", fontsize=18)
savefig(fig, "49_v2_11_region_search_pipeline.png")
plt.show()

## 14. Topology as leading specification

Topology is leading. Objective values are trailing summaries.

In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 5.0))
ax.axis("off")
left = [("Physics", 0.82), ("Resources", 0.68), ("Admissibility", 0.54), ("Topology Ωc", 0.40), ("Robustness", 0.26), ("Operation", 0.12)]
right = [("Objective", 0.70), ("Gradient", 0.52), ("Local optimum", 0.34)]
for label, yy in left:
    ax.add_patch(Rectangle((0.12, yy-0.045), 0.32, 0.09, fill=False, linewidth=2.0))
    ax.text(0.28, yy, label, ha="center", va="center", fontsize=11)
for i in range(len(left)-1):
    ax.annotate("", xy=(0.28, left[i+1][1]+0.052), xytext=(0.28, left[i][1]-0.052),
                arrowprops=dict(arrowstyle="->", linewidth=2.0))
for label, yy in right:
    ax.add_patch(Rectangle((0.62, yy-0.045), 0.28, 0.09, fill=False, linewidth=2.0))
    ax.text(0.76, yy, label, ha="center", va="center", fontsize=11)
for i in range(len(right)-1):
    ax.annotate("", xy=(0.76, right[i+1][1]+0.052), xytext=(0.76, right[i][1]-0.052),
                arrowprops=dict(arrowstyle="->", linewidth=2.0))
ax.text(0.28, 0.94, "Leading specification", ha="center", fontsize=13, weight="bold")
ax.text(0.76, 0.94, "Trailing optimization", ha="center", fontsize=13, weight="bold")
ax.text(0.5, 0.02, "Region-constrained search treats topology as the specification and objectives as summaries.",
        ha="center", fontsize=12)
ax.set_title("Topology Is the Leading Specification", fontsize=18)
savefig(fig, "49_v2_12_topology_leading_specification.png")
plt.show()

## 15. Engineering specification table

In [ ]:
spec = pd.DataFrame([
    {"Symbol": "Design", "Meaning": "candidate architecture or control policy", "Computation": "input to search"},
    {"Symbol": "Ω", "Meaning": "admissible region", "Computation": "thresholded specification mask"},
    {"Symbol": "Ωc", "Meaning": "largest connected admissible component", "Computation": "connected-component analysis"},
    {"Symbol": "d(∂Ωc)", "Meaning": "distance to failure boundary", "Computation": "distance transform"},
    {"Symbol": "x*", "Meaning": "maximum-margin operating point", "Computation": "argmax distance inside Ωc"},
    {"Symbol": "Fault-Tolerant Computation", "Meaning": "robust logical execution", "Computation": "selected from robust region"},
])
save_table(spec, "49_v2_engineering_specification")
fig, ax = plt.subplots(figsize=(14, 4.4))
ax.axis("off")
table = ax.table(cellText=spec.values, colLabels=spec.columns, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(9.2)
table.scale(1.05, 1.65)
for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.1)
    if r == 0:
        cell.set_text_props(weight="bold")
ax.set_title("Engineering Specification for Region-Constrained Design Search", fontsize=17, pad=18)
savefig(fig, "49_v2_13_engineering_specification.png")
plt.show()
spec

## 16. Export bundle

In [ ]:
review = {
    "architecture_metrics": architecture_metrics.to_dict(orient="records"),
    "design_evolution": evolution.to_dict(orient="records"),
    "engineering_specification": spec.to_dict(orient="records"),
}
with open(JS / "49_v2_review_bundle.json", "w", encoding="utf-8") as f:
    json.dump(review, f, indent=2)

zip_path = RES / "49_outputs_v2.zip"
files_to_zip = list(FIG.glob("49_v2_*.png")) + list(CSV.glob("49_v2_*.csv")) + list(JS.glob("49_v2_*.json"))

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

## Takeaways

- Region-constrained design search optimizes topology before selecting an operating point.
- The leading object is \( \Omega_c \), the largest connected admissible component.
- The maximum-margin operating point \(x^\*\) is selected after distance geometry is evaluated.
- Cost and control complexity belong in the objective, but they do not replace admissibility.
- Topology is the leading specification; objective functions are trailing summaries.

\[
\boxed{
\text{Design}
\rightarrow
\Omega
\rightarrow
\Omega_c
\rightarrow
d(\partial\Omega_c)
\rightarrow
x^\*
\rightarrow
\text{Fault-Tolerant Computation}
}
\]